In [ ]:
import pandas as pd
import os
from pathlib import Path

# -----------------------------
# PATHS
# -----------------------------
BASE = Path(__file__).resolve().parent
DATASET_DIR = BASE / "datasets"
OUTPUT_DIR = BASE / "standardized_data"
REPORT_DIR = BASE / "reports"

OUTPUT_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

files = {
    "billing": "Billing_payments.csv",
    "iiot": "iiot_smart_grid_dataset.csv",
    "oms": "OMS_data.csv",
    "maintenance": "asset_maintenance.csv",
    "scada": "scada_pipeline.csv"
}

# -----------------------------
# LOAD DATA
# -----------------------------
def load_csv(name):
    path = DATASET_DIR / files[name]
    if not path.exists():
        print(f"File not found: {path}")
        return pd.DataFrame()

    return pd.read_csv(path)


billing = load_csv("billing")
iiot = load_csv("iiot")
oms = load_csv("oms")
maintenance = load_csv("maintenance")
scada = load_csv("scada")


# -----------------------------
# HELPER FUNCTIONS
# -----------------------------
def clean_text(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype("string")
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
    return df


def clean_column_names(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[\[\]()]", "", regex=True)
        .str.replace(r"[^a-zA-Z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return df


# ============================================================
# 1. CUSTOMER MASTER
# ============================================================

customer_records = []

if not billing.empty:

    # Create one customer master record per utility
    cols = [
        "Utility.Number",
        "Utility.Name",
        "Utility.State",
        "Utility.Type",
        "Retail.Residential.Customers",
        "Retail.Commercial.Customers",
        "Retail.Industrial.Customers",
        "Retail.Transportation.Customers",
        "Retail.Total.Customers"
    ]

    available = [c for c in cols if c in billing.columns]

    customer = billing[available].copy()

    customer = clean_column_names(customer)

    # Standardize text
    text_cols = [
        "utility_name",
        "utility_state",
        "utility_type"
    ]

    customer = clean_text(customer, text_cols)

    # Rename columns
    rename_customer = {
        "utility_number": "customer_id",
        "utility_name": "utility_name",
        "utility_state": "state",
        "utility_type": "customer_type",
        "retail_residential_customers": "residential_customers",
        "retail_commercial_customers": "commercial_customers",
        "retail_industrial_customers": "industrial_customers",
        "retail_transportation_customers": "transportation_customers",
        "retail_total_customers": "total_customers"
    }

    customer.rename(columns=rename_customer, inplace=True)

    # Remove duplicate customer/utility IDs
    if "customer_id" in customer.columns:
        customer = customer.drop_duplicates(subset=["customer_id"])

    customer_records = customer


# ============================================================
# 2. ASSET MASTER
# ============================================================

if not maintenance.empty:

    asset = maintenance.copy()

    asset = clean_column_names(asset)

    # Standardize text
    asset = clean_text(
        asset,
        ["product_id", "type", "failure_type"]
    )

    rename_asset = {
        "udi": "asset_id",
        "product_id": "product_id",
        "type": "asset_type",
        "target": "failure_target",
        "failure_type": "failure_type"
    }

    asset.rename(columns=rename_asset, inplace=True)

    # Keep master-data columns
    asset_cols = [
        "asset_id",
        "product_id",
        "asset_type",
        "failure_target",
        "failure_type"
    ]

    asset_cols = [c for c in asset_cols if c in asset.columns]

    asset_master = asset[asset_cols].copy()

    # Remove duplicate assets
    if "asset_id" in asset_master.columns:
        asset_master = asset_master.drop_duplicates(subset=["asset_id"])

else:
    asset_master = pd.DataFrame()


# ============================================================
# 3. GRID MASTER
# ============================================================

grid_records = []

# SCADA grid information
if not scada.empty:

    grid = scada.copy()

    grid = clean_column_names(grid)

    if "segment_id" in grid.columns:
        grid_master = grid[["segment_id"]].drop_duplicates()

        grid_master.rename(
            columns={"segment_id": "grid_segment_id"},
            inplace=True
        )

    else:
        grid_master = pd.DataFrame()

else:
    grid_master = pd.DataFrame()


# Add utility information from Billing
if not billing.empty:

    utility = billing.copy()

    utility = clean_column_names(utility)

    utility_cols = [
        "utility_number",
        "utility_name",
        "utility_state",
        "utility_type"
    ]

    utility_cols = [
        c for c in utility_cols
        if c in utility.columns
    ]

    if utility_cols:

        utility_master = utility[utility_cols].drop_duplicates()

        utility_master.rename(
            columns={
                "utility_number": "utility_id",
                "utility_name": "utility_name",
                "utility_state": "state",
                "utility_type": "utility_type"
            },
            inplace=True
        )

        utility_master = clean_text(
            utility_master,
            ["utility_name", "state", "utility_type"]
        )

    else:
        utility_master = pd.DataFrame()

else:
    utility_master = pd.DataFrame()


# ============================================================
# 4. STANDARDIZE IIOT GRID CATEGORIES
# ============================================================

if not iiot.empty:

    iiot_grid = iiot.copy()

    iiot_grid = clean_column_names(iiot_grid)

    grid_cols = [
        "energy_source_type",
        "user_type",
        "weather_condition"
    ]

    grid_cols = [
        c for c in grid_cols
        if c in iiot_grid.columns
    ]

    if grid_cols:

        categories = []

        for col in grid_cols:

            values = (
                iiot_grid[col]
                .dropna()
                .astype(str)
                .str.strip()
                .str.title()
                .unique()
            )

            for value in values:
                categories.append({
                    "attribute": col,
                    "standardized_value": value
                })

        grid_categories = pd.DataFrame(categories)

    else:
        grid_categories = pd.DataFrame()

else:
    grid_categories = pd.DataFrame()


# ============================================================
# 5. VALIDATION
# ============================================================

validation = []


def validate_master(name, df, id_column):

    if df.empty:
        validation.append({
            "master": name,
            "records": 0,
            "duplicate_ids": 0,
            "missing_ids": 0,
            "status": "No data"
        })
        return

    duplicates = 0
    missing = 0

    if id_column in df.columns:

        duplicates = df[id_column].duplicated().sum()
        missing = df[id_column].isna().sum()

    validation.append({
        "master": name,
        "records": len(df),
        "duplicate_ids": duplicates,
        "missing_ids": missing,
        "status": "Completed"
    })


validate_master(
    "Customer Master",
    customer_records,
    "customer_id"
)

validate_master(
    "Asset Master",
    asset_master,
    "asset_id"
)

validate_master(
    "Grid Master",
    grid_master,
    "grid_segment_id"
)


validation_report = pd.DataFrame(validation)
# 6. STANDARDIZATION LOG
standardization_log = pd.DataFrame([
    {
        "dataset": "Billing",
        "standardization": "Column names converted to lowercase snake_case"
    },
    {
        "dataset": "Billing",
        "standardization": "Utility/customer fields renamed to master-data names"
    },
    {
        "dataset": "Billing",
        "standardization": "Whitespace removed from text fields"
    },
    {
        "dataset": "Billing",
        "standardization": "Duplicate customer/utility IDs removed"
    },
    {
        "dataset": "Asset Maintenance",
        "standardization": "UDI standardized as asset_id"
    },
    {
        "dataset": "Asset Maintenance",
        "standardization": "Product ID standardized as product_id"
    },
    {
        "dataset": "Asset Maintenance",
        "standardization": "Asset type standardized as asset_type"
    },
    {
        "dataset": "SCADA",
        "standardization": "segment_id standardized as grid_segment_id"
    },
    {
        "dataset": "IIoT Smart Grid",
        "standardization": "Grid categorical values trimmed and converted to title case"
    }
]
)
# 7. SAVE CSV FILES
if not customer_records.empty:
    customer_records.to_csv(
        OUTPUT_DIR / "customer_master.csv",
        index=False
    )
if not asset_master.empty:
    asset_master.to_csv(
        OUTPUT_DIR / "asset_master.csv",
        index=False
    )
if not grid_master.empty:
    grid_master.to_csv(
        OUTPUT_DIR / "grid_master.csv",
        index=False
    )

if not utility_master.empty:
    utility_master.to_csv(
        OUTPUT_DIR / "utility_master.csv",
        index=False
    )
# 8. EXCEL REPORT
report_file = REPORT_DIR / "Master_Data_Standardization_Report.xlsx"

with pd.ExcelWriter(report_file, engine="openpyxl") as writer:

    if not customer_records.empty:
        customer_records.to_excel(
            writer,
            sheet_name="Customer_Master",
            index=False
        )

    if not asset_master.empty:
        asset_master.to_excel(
            writer,
            sheet_name="Asset_Master",
            index=False
        )

    if not grid_master.empty:
        grid_master.to_excel(
            writer,
            sheet_name="Grid_Master",
            index=False
        )

    if not utility_master.empty:
        utility_master.to_excel(
            writer,
            sheet_name="Utility_Master",
            index=False
        )
    if not grid_categories.empty:
        grid_categories.to_excel(
            writer,
            sheet_name="Grid_Categories",
            index=False
        )

    validation_report.to_excel(
        writer,
        sheet_name="Validation_Report",
        index=False
    )
    standardization_log.to_excel(
        writer,
        sheet_name="Standardization_Log",
        index=False
    )
# 9. DISPLAY RESULTS
print("\nMASTER DATA STANDARDIZATION COMPLETED")
print("------------------------------------")

print(f"Customer Master : {len(customer_records)} records")
print(f"Asset Master    : {len(asset_master)} records")
print(f"Grid Master     : {len(grid_master)} records")

print("\nFiles created:")
print(f"- {OUTPUT_DIR / 'customer_master.csv'}")
print(f"- {OUTPUT_DIR / 'asset_master.csv'}")
print(f"- {OUTPUT_DIR / 'grid_master.csv'}")
print(f"- {REPORT_DIR / 'Master_Data_Standardization_Report.xlsx'}")